**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Channel Coding

[Digital Communications](./Digital_Communications.ipynb) got bits across a channel — *mostly*. [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) promised that rates below capacity can be error-**free**. This course builds the machinery that cashes that promise: Hamming codes, convolutional codes with Viterbi decoding (verified against brute force), and a look at the LDPC/polar codes in your phone.

## 1. Pre-requisites

- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4 (capacity).
- [Digital Communications](./Digital_Communications.ipynb) (the channel being protected).
- Binary arithmetic (XOR as addition mod 2).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Block Codes & Hamming* (~40 min)
**Goal:** add parity with structure: detect, then CORRECT errors; the Hamming (7,4) built from scratch.
**Feeds into:** Session 2 (convolutional codes & Viterbi).

---

## 2. Redundancy with Geometry

💡 **Intuition.** Repetition (send everything 3×) corrects single errors at rate 1/3 — brutal. Hamming's 1948 insight: parity bits can each *watch an overlapping subset* of data bits, so the pattern of failed checks (**the syndrome**) doesn't just announce an error — it spells out the *address* of the flipped bit. The (7,4) code corrects any single error at rate 4/7. Geometrically: codewords are spread out so every single-bit corruption still lands *nearest* its true codeword — minimum distance 3, correcting $\lfloor (d-1)/2 \rfloor = 1$ error.

In [2]:
# Hamming (7,4): generator and parity-check matrices over GF(2)
G = np.array([[1,0,0,0,1,1,0],
              [0,1,0,0,1,0,1],
              [0,0,1,0,0,1,1],
              [0,0,0,1,1,1,1]])
H = np.array([[1,1,0,1,1,0,0],
              [1,0,1,1,0,1,0],
              [0,1,1,1,0,0,1]])
assert not (G @ H.T % 2).any()                       # every codeword passes every check

def ham_encode(bits4): return bits4 @ G % 2
def ham_decode(word7):
    synd = word7 @ H.T % 2
    if synd.any():                                    # syndrome = column of H = error position
        err_pos = int(np.where((H.T == synd).all(1))[0][0])
        word7 = word7.copy(); word7[err_pos] ^= 1
    return word7[:4]

# every single-bit error on every message must be corrected — exhaustive check
ok = True
for msg_int in range(16):
    msg = np.array([(msg_int >> k) & 1 for k in range(4)])
    cw = ham_encode(msg)
    for e in range(7):
        rx = cw.copy(); rx[e] ^= 1
        ok &= (ham_decode(rx) == msg).all()
print("Hamming(7,4) corrects ALL 16×7 single-bit error cases:", ok)
assert ok

Hamming(7,4) corrects ALL 16×7 single-bit error cases: True


In [3]:
# BER on a binary symmetric channel: uncoded vs Hamming, simulated
def ber_sim(p_flip, n_msgs=30000, coded=True):
    errs = 0
    msgs = rng.integers(0, 2, (n_msgs, 4))
    for msg in msgs:
        if coded:
            tx = ham_encode(msg)
            rx = tx ^ (rng.random(7) < p_flip)
            errs += (ham_decode(rx) != msg).sum()
        else:
            rx = msg ^ (rng.random(4) < p_flip)
            errs += (rx != msg).sum()
    return errs / (n_msgs * 4)

ps = np.array([0.001, 0.003, 0.01, 0.03, 0.1])
plt.figure(figsize=(7.5, 3))
plt.loglog(ps, [ber_sim(p, coded=False) for p in ps], "o-", label="uncoded")
plt.loglog(ps, [ber_sim(p, coded=True) for p in ps], "s-", label="Hamming(7,4)")
plt.legend(); plt.xlabel("channel flip probability"); plt.ylabel("bit error rate")
plt.title("the coding gain: slope steepens because DOUBLE errors are now the failure mode")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2973931/3620951141.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Convolutional Codes & Viterbi* (~40 min)
**Goal:** encode with memory; decode with dynamic programming — verified against brute force.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (modern codes).

---

## 3. Codes with Memory

💡 **Intuition.** A convolutional encoder is an [FIR filter over GF(2)](./Foundations_of_Signal_Processing_1.ipynb): each input bit emits output bits that depend on the last $K$ inputs, entangling every bit with its neighbors. Decoding = finding the most likely *path* through the encoder's state **trellis** — and since path cost is additive, dynamic programming (**Viterbi**) finds the exact best path in linear time. It is [Bellman's principle](../Intro_Mach_Learn/Reinforcement_Learning.ipynb) applied to decoding — the same algorithm that powers speech recognition and DNA alignment.

In [4]:
# rate-1/2, K=3 convolutional code (the classic [7,5] octal generators)
G1, G2 = (1,1,1), (1,0,1)
def conv_encode(bits):
    state = (0, 0); out = []
    for b in list(bits) + [0, 0]:                    # tail bits flush the state
        reg = (b,) + state
        out += [sum(g*r for g, r in zip(G1, reg)) % 2,
                sum(g*r for g, r in zip(G2, reg)) % 2]
        state = (b, state[0])
    return np.array(out)

def viterbi(rx):
    n_steps = len(rx) // 2
    INF = 1e9
    cost = {(0, 0): 0.0}; back = []
    for t in range(n_steps):
        new_cost, bp = {}, {}
        for state, c in cost.items():
            for b in (0, 1):
                reg = (b,) + state
                o = (sum(g*r for g, r in zip(G1, reg)) % 2,
                     sum(g*r for g, r in zip(G2, reg)) % 2)
                branch = (o[0] != rx[2*t]) + (o[1] != rx[2*t+1])   # Hamming branch metric
                ns = (b, state[0])
                if c + branch < new_cost.get(ns, INF):
                    new_cost[ns] = c + branch; bp[ns] = (state, b)
        cost, _ = new_cost, back.append(bp)
    s = min(cost, key=cost.get)
    bits = []
    for bp in reversed(back):
        s, b = bp[s]
        bits.append(b)
    return np.array(bits[::-1][:n_steps-2])          # drop the tail

# ORACLE: Viterbi must equal brute-force maximum-likelihood over ALL 2^10 messages
msg = rng.integers(0, 2, 10)
tx = conv_encode(msg)
rx = tx ^ (rng.random(len(tx)) < 0.08)               # 8% bit flips

best, best_d = None, 1e9
for m_int in range(1024):
    cand = np.array([(m_int >> k) & 1 for k in range(10)])
    d = (conv_encode(cand) != rx).sum()
    if d < best_d: best, best_d = cand, d
vit = viterbi(rx)
print("Viterbi == brute-force ML decode:", (vit == best).all(), f"(both at distance {best_d})")
assert (vit == best).all()
print("decoded == transmitted:", (vit == msg).all())

Viterbi == brute-force ML decode: True (both at distance 2)
decoded == transmitted: True


In [5]:
# BER race at equal ENERGY per information bit (soft comparison, hard-decision decoding)
def ber_conv(p_flip, n_trials=1500, L=50):
    errs = tot = 0
    for _ in range(n_trials):
        m = rng.integers(0, 2, L)
        rx = conv_encode(m) ^ (rng.random(2*(L+2)) < p_flip)
        errs += (viterbi(rx) != m).sum(); tot += L
    return errs / tot

ps2 = [0.01, 0.03, 0.06, 0.1]
plt.figure(figsize=(7.5, 3))
plt.loglog(ps2, ps2, "o-", label="uncoded (BER = p)")
plt.loglog(ps2, [ber_conv(p) for p in ps2], "s-", label="conv. K=3 + Viterbi (rate 1/2)")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.xlabel("channel flip probability"); plt.ylabel("BER")
plt.title("memory + optimal decoding: orders of magnitude at moderate noise")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2973931/1756391977.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Modern Codes at a Glance* (~30 min)
**Goal:** why LDPC and polar codes closed the gap to Shannon — mechanisms, not implementations.
**Builds on:** Session 2.

---

## 4. Closing the Last dB

> ℹ️ **Survey session** — mechanisms and intuition; production LDPC/polar decoders are a course of their own and are *not* implemented here.

💡 **Intuition (LDPC).** A *sparse* random parity-check matrix — each bit in a few checks, each check watching a few bits — decoded by **belief propagation**: bits and checks pass probability messages on the graph ([graph signal processing](./Graph_Signal_Processing.ipynb) territory) until consistent. Gallager invented them in 1962; they waited 35 years for hardware. Your Wi-Fi and 5G data channels run them within ~0.5 dB of Shannon's limit.

💡 **Intuition (polar).** Chain two-bit butterflies recursively ([FFT structure](./Foundations_of_Signal_Processing_1.ipynb)!) and channels *polarize*: some synthetic bit-channels become nearly perfect, others nearly useless. Put data on the good ones, zeros on the bad — the first codes *provably* achieving capacity (Arıkan, 2009). 5G control channels use them.

**The map:**

| | Hamming | Convolutional | LDPC | Polar |
|---|---|---|---|---|
| Decoding | syndrome table | Viterbi (exact DP) | belief propagation (iterative) | successive cancellation |
| Gap to capacity | far | moderate | ~0.5 dB | → 0 (provable) |
| Where | teaching, ECC RAM | GPS, legacy comms | Wi-Fi/5G data | 5G control |

## 5. Conclusion

Syndromes spell out error addresses; trellises make optimal decoding a shortest path (verified against brute force); sparsity + message passing and recursive polarization close the last decibels to Shannon. The promise of [Information Theory S4](../Intro_Math/Information_Theory/Information_Theory.ipynb) is now an engineering fact you've simulated.

---
## Where next

- [Digital Communications](./Digital_Communications.ipynb) — the modem these codes ride.
- [Graph Signal Processing](./Graph_Signal_Processing.ipynb) — belief propagation's playing field.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — real packets carrying real codes.